# 18 Normal Second UPDATE Pilot

Controlled step only: run the second TRAIN-only Normal UPDATE qualification pilot using the eight RIB-prescreened candidates. This notebook does not access Validation/Holdout, change the 30-minute/60-minute policy, add CAIDA/RPKI, or train models.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
from bgp_incident_detection.normal_second_pilot import prepare_summary, run

PILOT = ROOT / 'data' / 'pilot'
prepare_summary()

## Execute Pilot

Downloads only the 91 HEAD-verified UPDATE files for the eight TRAIN candidates, parses them with `bgpdump`, builds 5-minute windows, applies the frozen 44 behavioral features, and qualifies each window as `NORMAL_VERIFIED` or `UNKNOWN`.

In [ ]:
summary = run()
summary

## Outputs

In [ ]:
execution = pd.read_csv(PILOT / 'normal_second_pilot_execution_manifest_v1.csv')
download_audit = pd.read_csv(PILOT / 'normal_second_pilot_download_audit_v1.csv')
raw_counts = pd.read_csv(PILOT / 'normal_second_pilot_raw_bgp_counts_v1.csv')
relevance = pd.read_csv(PILOT / 'normal_second_pilot_relevance_summary_v1.csv')
window_counts = pd.read_csv(PILOT / 'normal_second_pilot_window_counts_v1.csv')
qualification = pd.read_csv(PILOT / 'normal_second_pilot_qualification_results_v1.csv')
candidate_results = pd.read_csv(PILOT / 'normal_second_pilot_candidate_results_v1.csv')
comparison = pd.read_csv(PILOT / 'normal_pilot_comparison_v2.csv')

display(execution)
display(download_audit)
display(raw_counts)
display(relevance)
display(candidate_results)
display(comparison)

## Audit Checks

In [ ]:
features = pd.read_parquet(PILOT / 'normal_second_pilot_behavioral_features_v1.parquet')
assert len(execution) == 8
assert set(execution['partition']) == {'train'}
assert int(download_audit['planned_files'].sum()) == 91
assert int(download_audit['downloaded_files'].sum()) == 91
assert len(window_counts) == 48
assert len(features) == 48
assert set(qualification['final_state']) <= {'NORMAL_VERIFIED', 'UNKNOWN'}
assert (qualification['final_state'] == 'NORMAL_VERIFIED').sum() == 24
assert (qualification['final_state'] == 'UNKNOWN').sum() == 24
print('PASS: second Normal UPDATE pilot completed with frozen rules and TRAIN-only candidates.')